In [3]:
import numpy as np
from scipy.optimize import minimize

# Константы
SOUND_SPEED = 1125  # скорость звука (фут/с)
SAMPLE_RATE = 100000  # частота дискретизации (Гц)

# Позиции громкоговорителей (источников звука)
SPEAKER_POSITIONS = np.array([
    [0, 0, 10],
    [20, 0, 10],
    [20, 20, 10],
    [0, 20, 10]
])

def compute_time_delay(received_signal, reference_signal):
    """
    Вычисление временной задержки между сигналами
    с использованием взаимной корреляции
    """
    # Взаимная корреляция сигналов
    correlation = np.correlate(received_signal, reference_signal)
    
    # Индекс максимальной корреляции = временной сдвиг
    peak_index = np.argmax(correlation)
    
    # Преобразование индекса в секунды
    return peak_index / SAMPLE_RATE

def compute_distances_from_delays(delays):
    """
    Перевод временных задержек в расстояния
    """
    return [delay * SOUND_SPEED for delay in delays]

def residuals_squared(point, measured_distances):
    """
    Функция невязок для оптимизации методом наименьших квадратов
    point: предполагаемые координаты микрофона [x, y, z]
    measured_distances: измеренные расстояния до громкоговорителей
    """
    x, y, z = point
    errors = []
    
    for idx, (sx, sy, sz) in enumerate(SPEAKER_POSITIONS):
        # Евклидово расстояние от микрофона до громкоговорителя
        dx = x - sx
        dy = y - sy
        dz = z - sz
        calculated_distance = np.sqrt(dx*dx + dy*dy + dz*dz)
        
        # Разница между расчетным и измеренным расстоянием
        errors.append(calculated_distance - measured_distances[idx])
    
    return np.sum(np.square(errors))

# Загрузка данных
transmitter_signals = np.loadtxt("data_files/Transmitter.txt")  # сигналы с передатчиков (4xN)
receiver_signal = np.loadtxt("data_files/Receiver.txt")        # сигнал с приемника (1xN)

# Расчет временных задержек для всех громкоговорителей
time_delays = [
    compute_time_delay(receiver_signal, transmitter_signals[i])
    for i in range(len(SPEAKER_POSITIONS))
]

# Преобразование задержек в расстояния
measured_ranges = compute_distances_from_delays(time_delays)

# Начальное приближение для оптимизации
initial_estimate = [1.0, 1.0, 1.0]

# Минимизация методом L-BFGS-B
optimization_result = minimize(
    residuals_squared,
    initial_estimate,
    args=(measured_ranges,),
    method='L-BFGS-B'
)

# Извлечение результата
x_coord, y_coord, z_coord = optimization_result.x

print(f"Локализация микрофона завершена")
print(f"Координаты: X = {x_coord:.2f}, Y = {y_coord:.2f}, Z = {z_coord:.2f}")
print(f"Невязка: {optimization_result.fun:.6f}")

Локализация микрофона завершена
Координаты: X = 8.00, Y = 4.60, Z = 5.76
Невязка: 0.000034
